# Beyond Words — Colab inference server

This notebook loads your **existing A–D emotion checkpoints** from Google Drive and exposes a small authenticated HTTPS inference API. Your Streamlit interface runs locally and sends prediction requests to this Colab server.

**Important:** This is an emotion-only research demo. It does not implement sarcasm, sentiment, or sentiment-intent detection. The local app does not load PyTorch or Transformer checkpoints.

## Before you run
1. Put the saved model checkpoint folders in Google Drive. Keep the model files and tokenizer files together in each checkpoint folder.
2. Edit `PROJECT_DIR` in the setup cell if your Drive folder is not `MyDrive/Beyond-Words`.
3. Run all cells from top to bottom.
4. Use an available Colab GPU if possible.
5. Create an ngrok account and have its authtoken ready: https://dashboard.ngrok.com/get-started/your-authtoken

The notebook scans for folder paths containing `condA`/`condB`/`condC`/`condD` and a seed marker such as `seed_42`. Confirm the printed registry before launching. It keeps one model loaded at a time to limit GPU memory.


In [ ]:
# 1) Install server dependencies. The Transformers version matches the training/demo setup.
!pip -q install fastapi uvicorn pyngrok transformers==4.48.3 sentencepiece safetensors


In [ ]:
# 2) Mount Google Drive and configure the location of your project/checkpoints.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/Beyond-Words')  # EDIT this if your folder has a different name/location
SEARCH_DIRS = [PROJECT_DIR / 'models', PROJECT_DIR / 'outputs']

# If your checkpoints are elsewhere, add their parent folder to SEARCH_DIRS, e.g.
# SEARCH_DIRS.append(Path('/content/drive/MyDrive/MyModelExports'))

print('Project directory exists:', PROJECT_DIR.exists(), PROJECT_DIR)
for folder in SEARCH_DIRS:
    print('Search folder:', folder, '| exists:', folder.exists())


In [ ]:
# 3) Discover complete A-D checkpoints and build the registry used by the API.
import json, re
from collections import defaultdict

def has_model_weights(folder: Path) -> bool:
    patterns = ['model.safetensors', 'pytorch_model.bin', 'model-*.safetensors', 'pytorch_model-*.bin',
                'model.safetensors.index.json', 'pytorch_model.bin.index.json']
    return any(any(folder.glob(pattern)) for pattern in patterns)

candidates = defaultdict(list)
for search_root in SEARCH_DIRS:
    if not search_root.exists():
        continue
    for config_file in search_root.rglob('config.json'):
        model_dir = config_file.parent
        if not has_model_weights(model_dir):
            continue
        path_text = str(model_dir).replace('\\', '/')
        cond_match = re.search(r'cond([ABCD])', path_text, flags=re.IGNORECASE)
        seed_match = re.search(r'seed[_-]?(\d+)', path_text, flags=re.IGNORECASE)
        if not cond_match or not seed_match:
            continue
        condition = cond_match.group(1).upper()
        seed = int(seed_match.group(1))
        candidates[(condition, seed)].append(model_dir)

# Optional explicit overrides. Use these only if automatic discovery selects the wrong duplicate.
# Example: MANUAL_CHECKPOINTS = {'D': {42: '/content/drive/MyDrive/Beyond-Words/outputs/.../seed_42'}}
MANUAL_CHECKPOINTS = {}
registry = {'conditions': {}}
duplicates = []
for (condition, seed), paths in sorted(candidates.items()):
    # Prefer a saved seed folder over nested Trainer checkpoint-* folders; otherwise shortest path wins.
    chosen = sorted(paths, key=lambda p: (p.name.startswith('checkpoint-'), len(p.parts), str(p)))[0]
    if len(paths) > 1:
        duplicates.append((condition, seed, [str(p) for p in paths], str(chosen)))
    registry['conditions'].setdefault(condition, {})[str(seed)] = str(chosen)

for condition, seed_map in MANUAL_CHECKPOINTS.items():
    for seed, path in seed_map.items():
        p = Path(path)
        if not p.exists():
            raise FileNotFoundError(f'Manual checkpoint path does not exist: {p}')
        registry['conditions'].setdefault(condition, {})[str(seed)] = str(p)

registry_path = Path('/content/bw_checkpoint_registry.json')
registry_path.write_text(json.dumps(registry, indent=2), encoding='utf-8')
print('Checkpoint registry:')
for condition in ['A', 'B', 'C', 'D']:
    seed_map = registry['conditions'].get(condition, {})
    print(f'Condition {condition}:')
    for seed, path in sorted(seed_map.items(), key=lambda item: int(item[0])):
        print(f'  seed {seed}: {path}')
if duplicates:
    print('\nWARNING: duplicate condition/seed folders found. Review the selected path(s):')
    for item in duplicates:
        print(item)
if not registry['conditions']:
    raise RuntimeError('No checkpoints found. Edit PROJECT_DIR / SEARCH_DIRS, then rerun this cell.')
if 'D' not in registry['conditions']:
    print('WARNING: Condition D was not found. The local UI will default to the first available condition instead.')
print('\nRegistry saved to:', registry_path)


In [ ]:
# 4) Write the API service into this Colab runtime.
import base64
from pathlib import Path
API_SOURCE_B64 = (
    'IiIiRmFzdEFQSSBpbmZlcmVuY2Ugc2VydmljZSBmb3IgdGhlIEJleW9uZCBXb3JkcyBDb2xhYiBydW50aW1lLgoKVGhlIHNlcnZpY2UgbG9hZHMgb25lIHNlbGVjdGVkIGNoZWNrcG9pbnQgYXQgYSB0aW1lIHRvIHJlZHVjZSBDb2xhYiBHUFUgbWVtb3J5IHVzZS4KU2V0IEJXX0FQSV9LRVkgYW5kIEJXX1JFR0lTVFJZX1BBVEggYmVmb3JlIHN0YXJ0aW5nIHV2aWNvcm4uCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgZ2MKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCB0aHJlYWRpbmcKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmZyb20gZmFzdGFwaSBpbXBvcnQgRmFzdEFQSSwgSGVhZGVyLCBIVFRQRXhjZXB0aW9uCmZyb20gcHlkYW50aWMgaW1wb3J0IEJhc2VNb2RlbCwgRmllbGQKCkxBQkVMX09SREVSID0gWyJhbmdlciIsICJkaXNndXN0IiwgImZlYXIiLCAiam95IiwgInNhZG5lc3MiLCAic3VycHJpc2UiLCAibmV1dHJhbCJdCk1BWF9MRU4gPSAxMjgKUkVHSVNUUllfUEFUSCA9IFBhdGgob3MuZW52aXJvbi5nZXQoIkJXX1JFR0lTVFJZX1BBVEgiLCAiL2NvbnRlbnQvYndfY2hlY2twb2ludF9yZWdpc3RyeS5qc29uIikpCkFQSV9LRVkgPSBvcy5lbnZpcm9uLmdldCgiQldfQVBJX0tFWSIsICIiKQpDT05ESVRJT05fTkFNRVMgPSB7CiAgICAiQSI6ICJYTE0tUiwgRW5nbGlzaCBvbmx5IiwKICAgICJCIjogIlhMTS1SICsgRW1vU2VuIiwKICAgICJDIjogIkhpbmdSb0JFUlRhLCBFbmdsaXNoIG9ubHkiLAogICAgIkQiOiAiSGluZ1JvQkVSVGEgKyBFbW9TZW4iLAp9CgphcHAgPSBGYXN0QVBJKHRpdGxlPSJCZXlvbmQgV29yZHMgaW5mZXJlbmNlIEFQSSIsIHZlcnNpb249IjEuMC4wIikKX21vZGVsX2xvY2sgPSB0aHJlYWRpbmcuTG9jaygpCl9sb2FkZWRfa2V5OiB0dXBsZVtzdHIsIGludF0gfCBOb25lID0gTm9uZQpfdG9rZW5pemVyID0gTm9uZQpfbW9kZWwgPSBOb25lCl9kZXZpY2UgPSBOb25lCgoKZGVmIHJlYWRfcmVnaXN0cnkoKSAtPiBkaWN0W3N0ciwgZGljdFtzdHIsIHN0cl1dOgogICAgaWYgbm90IFJFR0lTVFJZX1BBVEguZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiQ2hlY2twb2ludCByZWdpc3RyeSBub3QgZm91bmQ6IHtSRUdJU1RSWV9QQVRIfSIpCiAgICBkYXRhID0ganNvbi5sb2FkcyhSRUdJU1RSWV9QQVRILnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHJldHVybiBkYXRhLmdldCgiY29uZGl0aW9ucyIsIHt9KQoKCmRlZiBjaGVja19hdXRoKGF1dGhvcml6YXRpb246IHN0ciB8IE5vbmUpIC0+IE5vbmU6CiAgICBpZiBub3QgQVBJX0tFWToKICAgICAgICByYWlzZSBIVFRQRXhjZXB0aW9uKHN0YXR1c19jb2RlPTUwMywgZGV0YWlsPSJTZXJ2ZXIgQVBJIGtleSBpcyBub3QgY29uZmlndXJlZC4iKQogICAgZXhwZWN0ZWQgPSBmIkJlYXJlciB7QVBJX0tFWX0iCiAgICBpZiBhdXRob3JpemF0aW9uICE9IGV4cGVjdGVkOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9NDAxLCBkZXRhaWw9Ik1pc3Npbmcgb3IgaW5jb3JyZWN0IGJlYXJlciBBUEkga2V5LiIpCgoKY2xhc3MgUHJlZGljdFJlcXVlc3QoQmFzZU1vZGVsKToKICAgIGNvbmRpdGlvbjogc3RyID0gRmllbGQocGF0dGVybj0iXltBQkNEXSQiKQogICAgc2VlZDogaW50ID0gRmllbGQoZ2U9MSkKICAgIHRleHQ6IHN0ciA9IEZpZWxkKG1pbl9sZW5ndGg9MSwgbWF4X2xlbmd0aD01MDAwKQogICAgdGVtcGVyYXR1cmU6IGZsb2F0ID0gRmllbGQoZGVmYXVsdD0xLjAsIGdlPTAuNSwgbGU9My4wKQoKCmRlZiBnZXRfbG9hZGVkX21vZGVsKGNvbmRpdGlvbjogc3RyLCBzZWVkOiBpbnQpOgogICAgZ2xvYmFsIF9sb2FkZWRfa2V5LCBfdG9rZW5pemVyLCBfbW9kZWwsIF9kZXZpY2UKICAgIGtleSA9IChjb25kaXRpb24sIHNlZWQpCiAgICBpZiBfbG9hZGVkX2tleSA9PSBrZXkgYW5kIF9tb2RlbCBpcyBub3QgTm9uZToKICAgICAgICByZXR1cm4gX3Rva2VuaXplciwgX21vZGVsLCBfZGV2aWNlCgogICAgcmVnaXN0cnkgPSByZWFkX3JlZ2lzdHJ5KCkKICAgIGNvbmRpdGlvbl9tb2RlbHMgPSByZWdpc3RyeS5nZXQoY29uZGl0aW9uKQogICAgaWYgbm90IGNvbmRpdGlvbl9tb2RlbHMgb3Igc3RyKHNlZWQpIG5vdCBpbiBjb25kaXRpb25fbW9kZWxzOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9NDA0LCBkZXRhaWw9ZiJDaGVja3BvaW50IG5vdCBhdmFpbGFibGUgZm9yIGNvbmRpdGlvbiB7Y29uZGl0aW9ufSwgc2VlZCB7c2VlZH0uIikKCiAgICAjIE9ubHkgb25lIGNoZWNrcG9pbnQgcmVtYWlucyByZXNpZGVudC4gU3dpdGNoaW5nIGNvbmRpdGlvbnMvc2VlZHMgZnJlZXMgdGhlIHByZXZpb3VzIG1vZGVsLgogICAgaWYgX21vZGVsIGlzIG5vdCBOb25lOgogICAgICAgIGRlbCBfbW9kZWwKICAgICAgICBkZWwgX3Rva2VuaXplcgogICAgICAgIF9tb2RlbCA9IE5vbmUKICAgICAgICBfdG9rZW5pemVyID0gTm9uZQogICAgICAgIF9sb2FkZWRfa2V5ID0gTm9uZQogICAgICAgIGdjLmNvbGxlY3QoKQogICAgICAgIHRyeToKICAgICAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBwYXNzCgogICAgcGF0aCA9IFBhdGgoY29uZGl0aW9uX21vZGVsc1tzdHIoc2VlZCldKQogICAgaWYgbm90IHBhdGguZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgSFRUUEV4Y2VwdGlvbihzdGF0dXNfY29kZT01MDAsIGRldGFpbD1mIkNoZWNrcG9pbnQgcGF0aCBpcyBubyBsb25nZXIgYXZhaWxhYmxlIGluIENvbGFiOiB7cGF0aH0iKQoKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgICAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b01vZGVsRm9yU2VxdWVuY2VDbGFzc2lmaWNhdGlvbiwgQXV0b1Rva2VuaXplcgoKICAgICAgICBfZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICAgICAgX3Rva2VuaXplciA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKHN0cihwYXRoKSwgbG9jYWxfZmlsZXNfb25seT1UcnVlKQogICAgICAgIF9tb2RlbCA9IEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24uZnJvbV9wcmV0cmFpbmVkKHN0cihwYXRoKSwgbG9jYWxfZmlsZXNfb25seT1UcnVlKQogICAgICAgIGlmIGludChfbW9kZWwuY29uZmlnLm51bV9sYWJlbHMpICE9IGxlbihMQUJFTF9PUkRFUik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICBmIkV4cGVjdGVkIGEgc2V2ZW4tb3V0cHV0IGNoZWNrcG9pbnQsIGZvdW5kIG51bV9sYWJlbHM9e19tb2RlbC5jb25maWcubnVtX2xhYmVsc30gYXQge3BhdGh9IgogICAgICAgICAgICApCiAgICAgICAgX21vZGVsLnRvKF9kZXZpY2UpCiAgICAgICAgX21vZGVsLmV2YWwoKQogICAgICAgIF9sb2FkZWRfa2V5ID0ga2V5CiAgICAgICAgcmV0dXJuIF90b2tlbml6ZXIsIF9tb2RlbCwgX2RldmljZQogICAgZXhjZXB0IEhUVFBFeGNlcHRpb246CiAgICAgICAgcmFpc2UKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIF9tb2RlbCA9IE5vbmUKICAgICAgICBfdG9rZW5pemVyID0gTm9uZQogICAgICAgIF9sb2FkZWRfa2V5ID0gTm9uZQogICAgICAgIGdjLmNvbGxlY3QoKQogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9NTAwLCBkZXRhaWw9ZiJDb3VsZCBub3QgbG9hZCB0aGUgc2VsZWN0ZWQgY2hlY2twb2ludDoge2V4Y30iKSBmcm9tIGV4YwoKCkBhcHAuZ2V0KCIvaGVhbHRoIikKZGVmIGhlYWx0aCgpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgdHJ5OgogICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgIGRldmljZSA9ICJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgZGV2aWNlID0gInVua25vd24iCiAgICByZXR1cm4geyJzdGF0dXMiOiAib2siLCAic2VydmljZSI6ICJiZXlvbmQtd29yZHMiLCAiZGV2aWNlIjogZGV2aWNlfQoKCkBhcHAuZ2V0KCIvbW9kZWxzIikKZGVmIG1vZGVscyhhdXRob3JpemF0aW9uOiBzdHIgfCBOb25lID0gSGVhZGVyKGRlZmF1bHQ9Tm9uZSkpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgY2hlY2tfYXV0aChhdXRob3JpemF0aW9uKQogICAgcmVnaXN0cnkgPSByZWFkX3JlZ2lzdHJ5KCkKICAgIG9yZGVyZWQgPSBbXQogICAgZm9yIGNvbmRpdGlvbiBpbiBbIkQiLCAiQyIsICJCIiwgIkEiXToKICAgICAgICBpZiBjb25kaXRpb24gaW4gcmVnaXN0cnk6CiAgICAgICAgICAgIG9yZGVyZWQuYXBwZW5kKHsKICAgICAgICAgICAgICAgICJjb25kaXRpb24iOiBjb25kaXRpb24sCiAgICAgICAgICAgICAgICAibmFtZSI6IENPTkRJVElPTl9OQU1FU1tjb25kaXRpb25dLAogICAgICAgICAgICAgICAgInNlZWRzIjogc29ydGVkKFtpbnQoc2VlZCkgZm9yIHNlZWQgaW4gcmVnaXN0cnlbY29uZGl0aW9uXS5rZXlzKCldKSwKICAgICAgICAgICAgfSkKICAgIHJldHVybiB7Im1vZGVscyI6IG9yZGVyZWQsICJsYWJlbHMiOiBMQUJFTF9PUkRFUiwgIm1heF9sZW5ndGgiOiBNQVhfTEVOfQoKCkBhcHAucG9zdCgiL3ByZWRpY3QiKQpkZWYgcHJlZGljdChyZXF1ZXN0OiBQcmVkaWN0UmVxdWVzdCwgYXV0aG9yaXphdGlvbjogc3RyIHwgTm9uZSA9IEhlYWRlcihkZWZhdWx0PU5vbmUpKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIGNoZWNrX2F1dGgoYXV0aG9yaXphdGlvbikKICAgIGNsZWFuX3RleHQgPSByZXF1ZXN0LnRleHQuc3RyaXAoKQogICAgaWYgbm90IGNsZWFuX3RleHQ6CiAgICAgICAgcmFpc2UgSFRUUEV4Y2VwdGlvbihzdGF0dXNfY29kZT00MjIsIGRldGFpbD0iVGV4dCBtdXN0IGNvbnRhaW4gYXQgbGVhc3Qgb25lIG5vbi13aGl0ZXNwYWNlIGNoYXJhY3Rlci4iKQoKICAgIHdpdGggX21vZGVsX2xvY2s6CiAgICAgICAgdG9rZW5pemVyLCBtb2RlbCwgZGV2aWNlID0gZ2V0X2xvYWRlZF9tb2RlbChyZXF1ZXN0LmNvbmRpdGlvbiwgcmVxdWVzdC5zZWVkKQogICAgICAgIHRyeToKICAgICAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgICAgIHVudHJ1bmNhdGVkID0gdG9rZW5pemVyKGNsZWFuX3RleHQsIGFkZF9zcGVjaWFsX3Rva2Vucz1UcnVlLCB0cnVuY2F0aW9uPUZhbHNlKVsiaW5wdXRfaWRzIl0KICAgICAgICAgICAgdHJ1bmNhdGVkID0gbGVuKHVudHJ1bmNhdGVkKSA+IE1BWF9MRU4KICAgICAgICAgICAgZW5jb2RlZCA9IHRva2VuaXplcigKICAgICAgICAgICAgICAgIGNsZWFuX3RleHQsCiAgICAgICAgICAgICAgICB0cnVuY2F0aW9uPVRydWUsCiAgICAgICAgICAgICAgICBtYXhfbGVuZ3RoPU1BWF9MRU4sCiAgICAgICAgICAgICAgICByZXR1cm5fdGVuc29ycz0icHQiLAogICAgICAgICAgICApCiAgICAgICAgICAgIHRva2VuX2NvdW50ID0gaW50KGVuY29kZWRbImlucHV0X2lkcyJdLnNoYXBlWzFdKQogICAgICAgICAgICBlbmNvZGVkID0ge25hbWU6IHRlbnNvci50byhkZXZpY2UpIGZvciBuYW1lLCB0ZW5zb3IgaW4gZW5jb2RlZC5pdGVtcygpfQogICAgICAgICAgICB3aXRoIHRvcmNoLmluZmVyZW5jZV9tb2RlKCk6CiAgICAgICAgICAgICAgICBsb2dpdHMgPSBtb2RlbCgqKmVuY29kZWQpLmxvZ2l0c1swXS5mbG9hdCgpCiAgICAgICAgICAgICAgICBwcm9iYWJpbGl0aWVzID0gdG9yY2guc29mdG1heChsb2dpdHMgLyBmbG9hdChyZXF1ZXN0LnRlbXBlcmF0dXJlKSwgZGltPS0xKS5jcHUoKS50b2xpc3QoKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgICAgICByYWlzZSBIVFRQRXhjZXB0aW9uKHN0YXR1c19jb2RlPTUwMCwgZGV0YWlsPWYiSW5mZXJlbmNlIGZhaWxlZDoge2V4Y30iKSBmcm9tIGV4YwoKICAgIHRvcF9pbmRpY2VzID0gc29ydGVkKHJhbmdlKGxlbihwcm9iYWJpbGl0aWVzKSksIGtleT1sYW1iZGEgaTogcHJvYmFiaWxpdGllc1tpXSwgcmV2ZXJzZT1UcnVlKQogICAgcmV0dXJuIHsKICAgICAgICAiY29uZGl0aW9uIjogcmVxdWVzdC5jb25kaXRpb24sCiAgICAgICAgInNlZWQiOiByZXF1ZXN0LnNlZWQsCiAgICAgICAgImxhYmVscyI6IExBQkVMX09SREVSLAogICAgICAgICJwcm9iYWJpbGl0aWVzIjogW2Zsb2F0KHgpIGZvciB4IGluIHByb2JhYmlsaXRpZXNdLAogICAgICAgICJwcmVkaWN0ZWRfbGFiZWwiOiBMQUJFTF9PUkRFUlt0b3BfaW5kaWNlc1swXV0sCiAgICAgICAgInRvcF9wcm9iYWJpbGl0eSI6IGZsb2F0KHByb2JhYmlsaXRpZXNbdG9wX2luZGljZXNbMF1dKSwKICAgICAgICAicnVubmVyX3VwX2xhYmVsIjogTEFCRUxfT1JERVJbdG9wX2luZGljZXNbMV1dLAogICAgICAgICJydW5uZXJfdXBfcHJvYmFiaWxpdHkiOiBmbG9hdChwcm9iYWJpbGl0aWVzW3RvcF9pbmRpY2VzWzFdXSksCiAgICAgICAgIm5fdG9rZW5zIjogdG9rZW5fY291bnQsCiAgICAgICAgInRydW5jYXRlZCI6IHRydW5jYXRlZCwKICAgICAgICAibWF4X2xlbmd0aCI6IE1BWF9MRU4sCiAgICAgICAgImRldmljZSI6IHN0cihkZXZpY2UpLAogICAgICAgICJ0ZW1wZXJhdHVyZSI6IGZsb2F0KHJlcXVlc3QudGVtcGVyYXR1cmUpLAogICAgfQo='
)
Path('/content/bw_api.py').write_bytes(base64.b64decode(API_SOURCE_B64))
print('API module written:', Path('/content/bw_api.py').exists())


## Authentication

The ngrok tunnel is publicly reachable, so the inference endpoints require a private bearer API key. The key below is generated for this runtime. Copy it into the local Streamlit sidebar; do not share it or publish screenshots containing it.

You need an ngrok authtoken. Paste it into the secure prompt below; it is not saved in the notebook source.


In [ ]:
# 5) Configure ngrok and create a fresh API key for this Colab session.
from getpass import getpass
import os, secrets
from pyngrok import ngrok

ngrok_token = getpass('Paste your ngrok authtoken (input hidden): ').strip()
if not ngrok_token:
    raise ValueError('An ngrok authtoken is required to expose this Colab API.')
ngrok.set_auth_token(ngrok_token)

API_KEY = secrets.token_urlsafe(32)
os.environ['BW_API_KEY'] = API_KEY
os.environ['BW_REGISTRY_PATH'] = str(registry_path)
print('A new API key has been generated for this runtime. Copy it from the final server cell output.')


In [ ]:
# 6) Start FastAPI and create an HTTPS tunnel. Keep this Colab runtime running during the demo.
import os, sys, time, subprocess, requests
from pyngrok import ngrok

# Stop leftovers if this cell is rerun.
try:
    ngrok.kill()
except Exception:
    pass
try:
    if 'server_process' in globals() and server_process.poll() is None:
        server_process.terminate()
        server_process.wait(timeout=5)
except Exception:
    pass

log_file = open('/content/bw_api.log', 'w')
server_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'bw_api:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content', env=os.environ.copy(), stdout=log_file, stderr=subprocess.STDOUT
)

ready = False
for _ in range(30):
    if server_process.poll() is not None:
        break
    try:
        check = requests.get('http://127.0.0.1:8000/health', timeout=2)
        if check.ok:
            ready = True
            break
    except requests.RequestException:
        time.sleep(1)

if not ready:
    print(Path('/content/bw_api.log').read_text(encoding='utf-8', errors='replace')[-6000:])
    raise RuntimeError('FastAPI did not start. Review the log above.')

tunnel = ngrok.connect(8000, 'http')
API_URL = tunnel.public_url.rstrip('/')
models_check = requests.get(API_URL + '/models', headers={'Authorization': f'Bearer {API_KEY}'}, timeout=20)
models_check.raise_for_status()

print('\nBEYOND WORDS API IS READY')
print('API URL (copy this to local Streamlit):', API_URL)
print('API KEY (copy this to local Streamlit; keep private):', API_KEY)
print('Available models:', models_check.json()['models'])
print('\nLeave this Colab runtime running while using the local app.')


In [ ]:
# 7) Optional lightweight endpoint test (does not load a checkpoint).
health_response = requests.get(API_URL + '/health', timeout=15)
print('Health:', health_response.json())
print('Model registry:', requests.get(API_URL + '/models', headers={'Authorization': f'Bearer {API_KEY}'}, timeout=15).json())


## Connect your local Streamlit app

1. Run `streamlit run app.py` from your local `local_app` folder.
2. In the app's left sidebar, paste the `API URL` and `API KEY` printed by cell 6.
3. Condition D is preselected when available. You can change A/B/C/D and choose any discovered seed.
4. The first prediction for a condition/seed may take longer because Colab loads that checkpoint on demand.

If the runtime disconnects or the ngrok URL expires, rerun cells 5 and 6, then paste the new URL and key into Streamlit. The API key and tunnel are temporary; do not hard-code them into a public repository.

### Important research caveats
- The live prediction comes from one selected seed. Performance cards show the experiment's mean ± standard deviation across three seeds.
- The probability outputs are not post-hoc calibrated.
- Condition D is a practical default for the Hinglish demo, not proof that adaptation reliably improves results.
- The app does not provide sarcasm, sentiment, or intent predictions.
- Text is transmitted to this Colab runtime over the tunnel. Do not submit sensitive or identifying text.
